# Pillar 3 — Dual MCQ Evaluation (Figures 5 and 6)

PathOPEN's MCQs are written to work both as open-ended standalone questions and as
traditional multiple choice. Two sub-pillars test the two modes.

**Figure 5 — Sub-pillar 3a, standalone validity (Benchmark 5)**
- **a** Mean score across the four Benchmark 5a/5b dimensions, three datasets
- **b** Score distribution for Standalone Completeness, per dataset

**Figure 6 — Sub-pillar 3b, shortcut resistance**
- **a** Zero-shot MCQ accuracy per model, three datasets
- **b** Full vs blind accuracy, and position-shuffle variance

## The result that needs careful framing

Removing the image costs models 19–28 points on PathMMU and PatchVQA but only **+1.7 to
−2.6 points on PathOPEN**. Read naively that says PathOPEN's MCQs are answerable from
text alone — the opposite of what 3b sets out to show.

The data says otherwise, and the figure is built to show why:

- **The image is being used.** 26% of PathOPEN predictions change when it is removed
  (InternVL: 170/229 identical, so 59 change). The changes cancel out rather than being
  absent.
- **The mechanism is option length.** PathOPEN options average 9.7 words against 4.1 for
  PathMMU, because §3.2 mandates options be complete standalone clinical statements. On
  PathMMU the image *fixes* 292 answers and *breaks* 41 — it is the only discriminating
  signal. On PathOPEN it fixes 20 and breaks 16.
- **This is the same property Benchmark 5a rewards.** PathOPEN scores highest on
  Standalone Completeness (1.891 vs 1.686 / 1.480). Options cannot be both complete
  standalone statements and uninformative without the image.

So Figure 6 reports the blind result rather than omitting it, and panel **b** shows the
fix/break decomposition that explains it. Dropping the diagnostic while keeping the
position-shuffle result — which came out favourably — is the version hardest to defend.

## Data status

InternVL has scored all three datasets for Benchmark 5 (6,971 calls). **Qwen is still
running** (2,770 of 6,971: PathOPEN complete, PathMMU partial, PatchVQA not started).
Figure 5 therefore reports InternVL for the cross-dataset comparison, and judge-vs-judge
agreement on PathOPEN only, where both judges are complete. Re-run this notebook when
Qwen finishes to add the remaining agreement numbers.

In [ ]:
import re

import os
import sys

sys.path.insert(0, os.path.abspath("."))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import nature_style as ns

ns.apply_style()

EVAL = os.path.join("..", "data_evaluation")
JUDGE_CKPT = os.path.join(EVAL, "vlm", "checkpoints")
ZEROSHOT = os.path.join(EVAL, "vlm_zeroshot", "analysis")
ZEROSHOT_CKPT = os.path.join(EVAL, "vlm_zeroshot", "checkpoints")

DATASETS = ["PathOPEN", "PathMMU", "PatchVQA"]
B5_CRITERIA = [
    ("Standalone Completeness", "Standalone\ncompleteness", "5a"),
    ("Visual Grounding", "Visual\ngrounding", "5a"),
    ("Distractor Standalone Plausibility", "Distractor\nplausibility", "5b"),
    ("Distractor Visual Grounding Error", "Distractor vis.\ngrounding error", "5b"),
]
print("style applied")

## Figure 5 data — Benchmark 5 scores

Read straight from the judge checkpoints. Mean plus a bootstrapped 95% CI, because §2.3
asks for "mean score ± 95% CI" and the CI is what distinguishes a real gap from noise
when three of the four dimensions differ by only ~0.2.

In [ ]:
import re

def load_benchmark5(judge_key):
    path = os.path.join(JUDGE_CKPT, f"{judge_key}_benchmark5.jsonl")
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


import json

internvl_b5 = load_benchmark5("internvl")
qwen_b5 = load_benchmark5("qwenvl")
print(f"internvl {len(internvl_b5)} records | qwenvl {len(qwen_b5)} (still running)")


def bootstrap_mean_ci(values, n_boot=2000, seed=0):
    """Percentile CI on a mean. Seeded so the figure is reproducible."""
    values = np.asarray(values, dtype=float)
    if len(values) < 3:
        return float(values.mean()) if len(values) else np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = [values[rng.integers(0, len(values), len(values))].mean() for _ in range(n_boot)]
    return float(values.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


rows = []
for criterion, label, sub_benchmark in B5_CRITERIA:
    for dataset in DATASETS:
        values = [r["scores"][criterion] for r in internvl_b5
                  if r["dataset"] == dataset and criterion in r["scores"]]
        if not values:
            continue
        mean, low, high = bootstrap_mean_ci(values)
        rows.append({"criterion": criterion, "label": label, "sub": sub_benchmark,
                     "dataset": dataset, "n": len(values),
                     "mean": mean, "ci_low": low, "ci_high": high})
benchmark5 = pd.DataFrame(rows)
print(benchmark5.pivot(index="label", columns="dataset", values="mean").round(3).to_string())

### Judge-vs-judge agreement, PathOPEN only

Benchmark 5 has **no human baseline** — §3.5.4 says so explicitly. Judge-vs-judge
agreement is the only reliability signal available, and it is a weak one: two models can
share a blind spot. Computed here on PathOPEN, the one dataset both judges have finished.

In [ ]:
import re

from sklearn.metrics import cohen_kappa_score

qwen_by_id = {r["item_id"]: r for r in qwen_b5}
agreement_rows = []
for criterion, label, _ in B5_CRITERIA:
    pairs = [(r["scores"][criterion], qwen_by_id[r["item_id"]]["scores"][criterion])
             for r in internvl_b5
             if r["dataset"] == "PathOPEN" and criterion in r["scores"]
             and r["item_id"] in qwen_by_id
             and criterion in qwen_by_id[r["item_id"]]["scores"]]
    if len(pairs) < 2:
        continue
    a, b = zip(*pairs)
    agreement_rows.append({
        "criterion": label.replace("\n", " "), "n": len(pairs),
        # -1 kept: it is a rubric level ("unable to evaluate"), not a missing value.
        "kappa": cohen_kappa_score(a, b, weights="quadratic", labels=[-1, 0, 1, 2]),
        "exact": 100 * np.mean([x == y for x, y in pairs]),
        "within1": 100 * np.mean([abs(x - y) <= 1 for x, y in pairs]),
    })
judge_agreement = pd.DataFrame(agreement_rows)
print(judge_agreement.round(3).to_string(index=False))

### Assemble Figure 5

In [ ]:
import re

import matplotlib.transforms as mtransforms

# One figure per pillar, three rows of two. The previous 2x3 layout put three panels in a
# 183 mm width, leaving each ~55 mm across - too narrow for six-model legends and
# multi-line tick labels, which is what made it read as cramped. Two per row nearly
# doubles the width of every panel; the height goes to 236 mm, just inside Nature's
# 247 mm maximum.
fig = plt.figure(figsize=ns.mm(ns.DOUBLE_COL_MM, 272))
outer = fig.add_gridspec(3, 1, height_ratios=[1, 1, 1], hspace=0.52)
grid = outer[0].subgridspec(1, 2, width_ratios=[1.35, 1], wspace=0.42)

# --- a: mean score per dimension, three datasets ---
ax = fig.add_subplot(grid[0, 0])
labels = [label for _, label, _ in B5_CRITERIA]
x = np.arange(len(labels))
width = 0.26
for offset, dataset in zip((-width, 0, width), DATASETS):
    subset = benchmark5[benchmark5.dataset == dataset].set_index("label").loc[labels]
    errors = np.vstack([subset["mean"] - subset.ci_low, subset.ci_high - subset["mean"]])
    ax.bar(x + offset, subset["mean"], width, yerr=errors, capsize=1.2,
           color=ns.DATASET_COLORS[dataset], label=dataset,
           error_kw={"lw": 0.5, "capthick": 0.5})
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=6.5)
ax.set_ylabel("Mean score")
ax.set_ylim(0, 2.15)
ax.axhline(2, ls=":", lw=0.4, color="#999999")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.17), ncol=3, fontsize=6.5)
# 5a vs 5b split: the left pair rates the CORRECT option, the right pair the distractors.
ax.axvline(1.5, lw=0.4, color="#CCCCCC", ls="--")
# Group brackets under the tick labels: criteria 0-1 rate the CORRECT option (5a),
# criteria 2-3 the DISTRACTORS (5b). Drawn in a blended transform so the span is set in
# data coordinates while the depth stays fixed in axes fractions, below the tick text.
group_transform = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
for (start, end, group_label) in [(-0.42, 1.42, "5a: correct option"),
                                  (1.58, 3.42, "5b: distractors")]:
    ax.plot([start, start, end, end], [-0.17, -0.20, -0.20, -0.17],
            transform=group_transform, lw=0.6, color="#666666",
            clip_on=False, solid_capstyle="butt")
    ax.text((start + end) / 2, -0.215, group_label, transform=group_transform,
            ha="center", va="top", fontsize=6.5, color="#666666")
ns.panel_label_below(ax, "a", dy=-0.30)

# --- b: score distribution for Standalone Completeness ---
ax = fig.add_subplot(grid[0, 1])
bottom = np.zeros(len(DATASETS))
for score in (2, 1, 0, -1):
    fractions = []
    for dataset in DATASETS:
        values = [r["scores"]["Standalone Completeness"] for r in internvl_b5
                  if r["dataset"] == dataset and "Standalone Completeness" in r["scores"]]
        fractions.append(100 * np.mean([v == score for v in values]) if values else 0)
    fractions = np.array(fractions)
    ax.bar(np.arange(len(DATASETS)), fractions, 0.6, bottom=bottom,
           color=ns.SCORE_COLORS[score], label=str(score),
           edgecolor="white", linewidth=0.4)
    for index, (value, base) in enumerate(zip(fractions, bottom)):
        if value > 7:
            ax.text(index, base + value / 2, f"{value:.0f}", ha="center", va="center",
                    fontsize=6.5, color="white" if score == 2 else "black")
    bottom += fractions
ax.set_xticks(np.arange(len(DATASETS)))
ax.set_xticklabels(DATASETS, fontsize=6.5, rotation=20, ha="right")
ax.set_ylabel("Options (%)")
ax.set_ylim(0, 100)
ax.set_title("Standalone completeness", fontsize=7, pad=3)
ax.legend(title="Score", loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=6.5,
          title_fontsize=6.5)
ns.panel_label_below(ax, "b")

# --- c: judge-vs-judge agreement (PathOPEN, the only complete pair) ---
# c moves to row 3 beside f; row 2 is reserved for d and e, which share
# the six-model colour scheme and therefore a single legend.
row3 = outer[2].subgridspec(1, 2, width_ratios=[1, 1.15], wspace=0.48)
ax = fig.add_subplot(row3[0, 0])
y = np.arange(len(judge_agreement))
ax.barh(y, judge_agreement.kappa, 0.5, color=ns.OKABE_ITO["green"])
for index, row in judge_agreement.iterrows():
    ax.text(row.kappa + 0.02, index, f"{row.kappa:.2f}", va="center", fontsize=6.5)
ax.set_yticks(y)
ax.set_yticklabels([c.replace(" ", "\n", 1) for c in judge_agreement.criterion], fontsize=6.5)
ax.set_xlabel("Judge-vs-judge $\\kappa$", fontsize=7)
ax.set_xlim(0, 0.75)
ax.invert_yaxis()
# No human baseline exists for Benchmark 5; say so on the figure so the number is not
# mistaken for validation against ground truth.
ax.text(0.97, 0.05, "PathOPEN only — no\nhuman baseline",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=6.5, color="#666666")
ns.panel_label_below(ax, "c")

# Saved after the sub-pillar 3b panels are added in the next cell.
print("panels a-c drawn")

## Figure 6 — shortcut resistance

Panel **b** carries the decomposition. A single "image gain" number hides the mechanism:
on PathOPEN the image changes plenty of answers, but roughly as many for the worse as for
the better. Showing fixes and breaks separately is what turns an apparent null into an
interpretable result.

In [ ]:
import re

shortcut = pd.read_csv(os.path.join(ZEROSHOT, "mcq_shortcut.csv"))
print(shortcut[["model", "dataset", "full_accuracy", "blind_accuracy",
                "image_gain", "shuffle_spread"]].to_string(index=False))


def fix_break_counts(model_key):
    """Per dataset: how many items the image FIXED vs BROKE.

    A net gain near zero can mean the image is ignored, or that it helps and hurts in
    equal measure. Only the decomposition distinguishes those, and they are very
    different claims about a dataset."""
    path = os.path.join(ZEROSHOT_CKPT, f"{model_key}_mcq.jsonl")
    with open(path) as f:
        records = [json.loads(line) for line in f if line.strip()]
    out = {}
    for dataset in DATASETS:
        full = {r["task_id"]: r for r in records
                if r["dataset"] == dataset and r["condition"] == "full"}
        blind = {r["task_id"]: r for r in records
                 if r["dataset"] == dataset and r["condition"] == "blind"}
        shared = set(full) & set(blind)
        fixed = sum(1 for i in shared
                    if full[i]["predicted_index"] == full[i]["correct_index"]
                    and blind[i]["predicted_index"] != blind[i]["correct_index"])
        broke = sum(1 for i in shared
                    if full[i]["predicted_index"] != full[i]["correct_index"]
                    and blind[i]["predicted_index"] == blind[i]["correct_index"])
        changed = sum(1 for i in shared
                      if full[i]["predicted_index"] != blind[i]["predicted_index"])
        out[dataset] = {"n": len(shared), "fixed": fixed, "broke": broke,
                        "changed_pct": 100 * changed / len(shared) if shared else 0}
    return out


decomposition = {m: fix_break_counts(m) for m in ns.MODEL_ORDER}
for model, per_dataset in decomposition.items():
    summary = "  ".join(f"{d}: +{v['fixed']}/-{v['broke']} ({v['changed_pct']:.0f}% changed)"
                        for d, v in per_dataset.items())
    print(f"{model:12s} {summary}")

In [ ]:
import re

from matplotlib.transforms import Bbox

row2 = outer[1].subgridspec(1, 2, width_ratios=[1.15, 1], wspace=0.48)

# --- d: full-context accuracy ---
ax = fig.add_subplot(row2[0, 0])
x = np.arange(len(DATASETS))
width = 0.13
for index, model in enumerate(ns.MODEL_ORDER):
    subset = shortcut[shortcut.model == model].set_index("dataset").reindex(DATASETS)
    ax.bar(x + (index - 2.5) * width, subset.full_accuracy, width,
           color=ns.MODEL_COLORS[model], label=ns.MODEL_LABELS[model])
# Chance differs per dataset (option counts range 2-16), so it is drawn per dataset
# rather than as one line - "42% accuracy" means different things at 5 and at 16 options.
for index, dataset in enumerate(DATASETS):
    chance = shortcut[shortcut.dataset == dataset].chance.iloc[0]
    ax.plot([index - 0.42, index + 0.42], [chance, chance], ls="--", lw=0.6, color="#666666")
ax.text(2.45, shortcut[shortcut.dataset == "PatchVQA"].chance.iloc[0] + 1.5, "chance",
        fontsize=6.5, color="#666666")
ax.set_xticks(x)
ax.set_xticklabels(DATASETS, fontsize=7)
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 78)
ax_d = ax
ns.panel_label_below(ax, "d")

# --- e: image gain as a sorted dumbbell plot ---
# Was 18 vertical segments grouped by dataset and coloured by model: the two NEGATIVE
# gains - models that do WORSE when shown the image - were indistinguishable from short
# positive ones, and the reader had to compare segment lengths across three clusters.
# Sorting by gain and colouring by DIRECTION makes both readable at a glance.
ax_e = fig.add_subplot(row2[0, 1])
ax = ax_e
gains = (shortcut[["model", "dataset", "full_accuracy", "blind_accuracy", "image_gain"]]
         .sort_values("image_gain").reset_index(drop=True))
for row_index, row in gains.iterrows():
    helps = row.image_gain >= 0
    colour = ns.OKABE_ITO["green"] if helps else ns.OKABE_ITO["vermillion"]
    ax.plot([row.blind_accuracy, row.full_accuracy], [row_index] * 2,
            lw=1.0, color=colour, alpha=0.9, zorder=1, solid_capstyle="round")
    # Open circle = blind, filled = with the image; the fill is what the image adds.
    ax.plot(row.blind_accuracy, row_index, "o", ms=2.8, mfc="white", mec=colour,
            mew=0.8, zorder=2)
    ax.plot(row.full_accuracy, row_index, "o", ms=3.2, color=colour, mec="white",
            mew=0.4, zorder=3)

ax.set_yticks(range(len(gains)))
ax.set_yticklabels(
    [f"{re.sub(r'-[0-9.]+B$', '', ns.MODEL_LABELS[row.model])} · {row.dataset}"
     for _, row in gains.iterrows()], fontsize=5.5)
for tick, (_, row) in zip(ax.get_yticklabels(), gains.iterrows()):
    if row.dataset == "PathOPEN":
        tick.set_color(ns.DATASET_COLORS["PathOPEN"])
ax.set_xlabel("Accuracy (%)", fontsize=7)
ax.set_xlim(0, 78)
ax.set_ylim(-0.8, len(gains) - 0.2)
ax.invert_yaxis()
ax.set_title("blind (\u25cb) \u2192 with image (\u25cf)", fontsize=7, pad=3)
# The two rows where the image HURTS - the finding the old grouping buried.
for row_index, row in gains.iterrows():
    if row.image_gain < 0:
        ax.text(row.blind_accuracy + 1.5, row_index, f"{row.image_gain:+.1f}",
                va="center", ha="left", fontsize=5.5,
                color=ns.OKABE_ITO["vermillion"])
ax.text(77, 8.6, "PathOPEN rows (orange)\ncluster at the low-gain end",
        ha="right", va="center", fontsize=5.8, color="#666666", linespacing=1.25)
ns.panel_label_below(ax, "e")

# --- f: fix vs break, the mechanism behind panel e ---
ax = fig.add_subplot(row3[0, 1])
width = 0.35
for index, dataset in enumerate(DATASETS):
    fixed = np.mean([decomposition[m][dataset]["fixed"] for m in ns.MODEL_ORDER])
    broke = np.mean([decomposition[m][dataset]["broke"] for m in ns.MODEL_ORDER])
    ax.bar(index - width / 2, fixed, width, color=ns.OKABE_ITO["green"],
           label="Image fixes" if index == 0 else None)
    ax.bar(index + width / 2, broke, width, color=ns.OKABE_ITO["vermillion"],
           label="Image breaks" if index == 0 else None)
    changed = np.mean([decomposition[m][dataset]["changed_pct"] for m in ns.MODEL_ORDER])
    ax.text(index, max(fixed, broke) + 16, f"{changed:.0f}%\nchange",
            ha="center", fontsize=6, color="#666666")
ax.set_xticks(x)
ax.set_xticklabels(DATASETS, fontsize=7)
ax.set_ylabel("Items (mean over models)")
# Headroom for the "% of answers change" notes, which are placed in data coordinates
# above the taller of each pair of bars.
ax.set_ylim(top=ax.get_ylim()[1] * 1.28)
ns.legend_outside(ax, "above", ncol=2, fontsize=6.5,
                  bbox_to_anchor=(0.5, 1.02))
ns.panel_label_below(ax, "f")

# d and e plot the same six models in the same colours, so one legend centred over the
# pair replaces the key that previously sat on d alone - it recovers d's headroom and
# makes explicit that the colours mean the same thing in both panels.
fig.canvas.draw()
_r = fig.canvas.get_renderer()
_pair = Bbox.union([ax_d.get_window_extent(_r), ax_e.get_window_extent(_r)])
_handles = [plt.Line2D([], [], color=ns.MODEL_COLORS[m], lw=2.2, label=ns.MODEL_LABELS[m])
            for m in ns.MODEL_ORDER]
_top = max(ax_d.get_position().y1, ax_e.get_position().y1)
fig.legend(_handles, [h.get_label() for h in _handles],
           loc="lower center",
           bbox_to_anchor=((_pair.x0 + _pair.x1) / 2 / fig.get_window_extent().width,
                           _top + 0.006),
           ncol=6, frameon=False, fontsize=6.5, columnspacing=1.1, handletextpad=0.4)

_shared = fig.legends[-1]
ns.lift_legend_above_titles(fig, _shared, [ax_d, ax_e])

paths = ns.save(fig, "fig03_pillar3_dual_mcq")
print("wrote:", paths)
plt.show()

## Draft caption

**Fig. 3 | PathOPEN's MCQ options function as standalone clinical statements, and its
near-zero blind-accuracy gap reflects that design rather than an ignored image.**
**a**, Mean Benchmark 5 score across the four scoring dimensions for the three MCQ
datasets, rated by the InternVL judge. Error bars are bootstrapped 95% confidence
intervals (2,000 resamples). Group brackets beneath the axis mark the two halves: 5a
dimensions rate the correct option, 5b the distractors. PathOPEN scores highest on three
of four dimensions; Visual Grounding does not separate the datasets.
**b**, Score distribution for Standalone Completeness. PathOPEN options are rated the
maximum score more often than either comparator, consistent with the requirement (§3.2)
that options be written as complete clinical statements rather than short labels.
**c**, Judge-vs-judge agreement (quadratic-weighted Cohen's κ) between the two VLM judges
on PathOPEN. Benchmark 5 has **no human baseline** (§3.5.4), so this is a reliability
signal only, not validation against ground truth: two models may share a blind spot.
PathMMU and PatchVQA are omitted because the second judge's run was incomplete at the
time of writing.
**d**, Full-context zero-shot accuracy for six VLMs across the three MCQ datasets. Dashed
lines give the per-dataset chance level, which differs because option counts range from
2 to 16.
**e**, Image gain as a dumbbell plot: one row per model × dataset pair, ordered by gain
(smallest at top). The open circle is blind accuracy (image withheld), the filled circle
accuracy with the image, and the connector is the difference. Colour encodes **direction**
— green where the image helps, red where it hurts — and row labels for PathOPEN pairs are
printed in orange. MedGemma loses 2.6 pp and Quilt-LLaVA 0.4 pp on PathOPEN when shown
the image; all six PathOPEN pairs sit at the low-gain end (−2.6 to +3.9 pp) while the
largest gains, up to +27.7 pp, are all PathMMU and PatchVQA.
**f**, Decomposition of that difference: the number of items the image corrects (green)
versus disrupts (red), averaged over models, with the percentage of predictions that
change at all printed above each pair. Pooled across the six models the image corrects
4.0× more answers than it disrupts on PathMMU and 3.8× on PatchVQA, but only 1.3× on
PathOPEN, while 24% of PathOPEN predictions (11–45% by model) still change between
conditions. PathOPEN's near-zero *net* image gain therefore reflects image evidence
correcting and disrupting in similar measure, not the image being ignored.

**Layout note.** Panels are arranged in three rows of two: **a**, **b** | **d**, **e** |
**c**, **f**. Panels **d** and **e** share the six-model colour scheme and therefore a
single legend centred above the pair.

---

### Numbers a caption must not get wrong

| quantity | value |
|---|---|
| B5 Standalone Completeness (PathOPEN / PathMMU / PatchVQA) | 1.891 / 1.686 / 1.480 |
| B5 Visual Grounding | 1.852 / 1.885 / 1.810 — no separation |
| B5 Distractor Plausibility | 1.189 / 0.984 / 1.100 |
| image gain, PathMMU / PatchVQA / PathOPEN | +2.2 to +27.7 / +4.1 to +24.8 / −2.6 to +3.9 pp |
| negative image gain | MedGemma −2.6, Quilt-LLaVA −0.4 (both PathOPEN) |
| PathOPEN predictions changed by image | 24% mean (11–45% by model) |
| fix:break ratio, PathOPEN / PathMMU / PatchVQA | 1.3× / 4.0× / 3.8× |
| PathOPEN option length vs PathMMU | 9.7 vs 4.1 words |
| shuffle spread, all models | 0.3 – 7.4 pp |

### What the text must say, and must not omit

**Report the blind-accuracy result.** It is the shortcut-resistance test §2.1 commits to
in writing. Omitting it while keeping the position-shuffle result — which came out
favourably — is not defensible if a reviewer asks whether text-only answerability was
tested, and PathMMU/PatchVQA blind numbers would still appear in the same figure.

Suggested framing:

> PathOPEN's blind accuracy is higher than comparators', reflecting its design mandate
> that MCQ options be complete standalone clinical statements (§3.2) rather than short
> labels — the same property validated by Benchmark 5a. Image removal alters 24% of
> PathOPEN predictions, confirming visual grounding is used; the near-zero *net* change
> reflects image evidence correcting and disrupting answers in roughly equal measure
> (1.3× more corrections than disruptions), whereas on PathMMU it corrects 4.0× more
> often than it disrupts.

**Regenerate panel c when the second judge finishes.** It currently covers PathOPEN only
(1,145 items scored by both judges); Qwen's Benchmark 5 run was incomplete at the time of
writing.
